# seeds_ZD3D_fixed

Two more seeds for the fixed weight 1e-3. Benchmark ZD3D, 2 run(s), about 60 min on a T4 GPU.

Files needed in `MyDrive/SAW_ablation/`: `zd3d_core.py`, `saw_ablation.py`.

Results are saved to `MyDrive/SAW_ablation/seeds_ZD3D_fixed/` after every run; re-running the notebook skips finished runs.

In [ ]:
import os, sys, csv, time
import numpy as np, torch
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/SAW_ablation'
sys.path.insert(0, PROJECT_DIR)
import zd3d_core as core
import saw_ablation as sa

EXP, BENCH = 'seeds_ZD3D_fixed', 'ZD3D'
OUT = f'{PROJECT_DIR}/{EXP}'
os.makedirs(OUT, exist_ok=True)
device = 'cuda'
print(torch.cuda.get_device_name(0))

In [ ]:
RUNS = [{'mode': 'fixed', 'seed': 45, 'w_eik': 0.001}, {'mode': 'fixed', 'seed': 46, 'w_eik': 0.001}]
CSV_PATH = f'{OUT}/{EXP}.csv'
grid = core.Grid(n=sa.GRID_N)
PHI_E = None

In [ ]:
def key(r):
    return (r['mode'], str(r['seed']), str(r.get('q', '')), str(r.get('w_eik', '')))

done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        done = {(r['mode'], r['seed'], r['q'], r['w_eik']) for r in csv.DictReader(f)}
todo = [r for r in RUNS if key(r) not in done]
print(f'{len(RUNS) - len(todo)} done, {len(todo)} to go')

for run in todo:
    run = dict(run)
    print(run['mode'], 'seed', run['seed'], flush=True)
    row, tag = sa.run_one(core, BENCH, run, device, OUT, grid, PHI_E)
    new = not os.path.exists(CSV_PATH)
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"  rel L2 {row['avg_rel_L2']:.4f}%  MAPE {row['avg_vol_MAPE']:.4f}%  "
          f"final w {row['w_eik_final']}  {row['minutes']:.1f} min", flush=True)

In [ ]:
for r in csv.DictReader(open(CSV_PATH)):
    extra = f"  F_fill {float(r['avg_slot_filled']):.2f}%" if 'avg_slot_filled' in r else ''
    print(f"{r['mode']:7s} seed {r['seed']}  rel L2 {float(r['avg_rel_L2']):.4f}%  "
          f"MAPE {float(r['avg_vol_MAPE']):.4f}%{extra}  final w {r['w_eik_final']}")